# Exploring the ARMD microbiology data

This notebook documents my initial exploratory analysis of the Stanford ARMD
datasets. I want to understand how the data is organized and identify
any issues that will affect the ELT pipeline and susceptibility analysis.

## Data source

[Stanford ARMD dataset on Dryad](https://datadryad.org/dataset/doi:10.5061/dryad.jq2bvq8kp)

I downloaded the following files from the **April 11, 2025 version**
of the dataset:

- `microbiology_cultures_cohort.csv`
- `microbiology_cultures_demographics.csv`
- `microbiology_cultures_ward_info.csv`

The downloaded files are stored in `data/raw/`.

## Questions to investigate

- What does one row represent in each file?
- Which identifiers connect the three datasets?
- How are organisms and antibiotic susceptibility results recorded?
- How are negative cultures and missing results represented?
- Are there repeated records, and what explains them?



In [1]:
from pathlib import Path
import pandas as pd

Path.cwd()

WindowsPath('c:/Users/clark/OneDrive/Documents/Python Scripts/ELT-Microbiology/notebooks/source_exploration')

## Cultures dataset

Inspect the first 10,000 rows of the cultures dataset: `microbiology_cultures_cohort.csv`

In [2]:
data_dir = Path("../../data/raw")

cultures = pd.read_csv(
    data_dir / "microbiology_cultures_cohort.csv",
    nrows=10_000
)

cultures.head()

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,ordering_mode,culture_description,was_positive,organism,antibiotic,susceptibility
0,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
1,JC1713666,131300064625,697566032,2020-12-27 00:40:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
2,JC1669304,131272997044,620809641,2019-07-02 19:54:00+00:00,Inpatient,BLOOD,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
3,JC1697441,131208305006,510635146,2016-12-01 13:59:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
4,JC600786,131344486993,834128837,2022-11-18 19:35:00+00:00,Outpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible


In [14]:
cultures.columns.tolist()


['anon_id',
 'pat_enc_csn_id_coded',
 'order_proc_id_coded',
 'order_time_jittered_utc',
 'ordering_mode',
 'culture_description',
 'was_positive',
 'organism',
 'antibiotic',
 'susceptibility']

In [15]:
cultures.shape

(10000, 10)

In [16]:
cultures.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   anon_id                  10000 non-null  str  
 1   pat_enc_csn_id_coded     10000 non-null  int64
 2   order_proc_id_coded      10000 non-null  int64
 3   order_time_jittered_utc  10000 non-null  str  
 4   ordering_mode            10000 non-null  str  
 5   culture_description      10000 non-null  str  
 6   was_positive             10000 non-null  int64
 7   organism                 10000 non-null  str  
 8   antibiotic               10000 non-null  str  
 9   susceptibility           10000 non-null  str  
dtypes: int64(3), str(7)
memory usage: 781.4 KB


In [18]:
with pd.option_context("display.max_columns", None):
    display(cultures.head(30))

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,ordering_mode,culture_description,was_positive,organism,antibiotic,susceptibility
0,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
1,JC1713666,131300064625,697566032,2020-12-27 00:40:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
2,JC1669304,131272997044,620809641,2019-07-02 19:54:00+00:00,Inpatient,BLOOD,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
3,JC1697441,131208305006,510635146,2016-12-01 13:59:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
4,JC600786,131344486993,834128837,2022-11-18 19:35:00+00:00,Outpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
5,JC1558041,131255756247,573086383,2018-09-11 04:30:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
6,JC1783302,131294514927,681936151,2020-08-21 11:08:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
7,JC934238,131295466446,684087246,2020-10-08 18:12:00+00:00,Inpatient,BLOOD,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
8,JC1523245,131267745763,609754703,2019-06-02 20:15:00+00:00,Outpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
9,JC1036903,131284302383,655477900,2020-02-03 21:09:00+00:00,Outpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible


### Checking rows per culture

In [21]:
order_counts = cultures["order_proc_id_coded"].value_counts()

order_counts.head(10)

order_proc_id_coded
723621139    5
648526617    5
678700595    5
551244230    5
588660636    4
541839436    4
790680642    4
764510010    4
588963204    4
538327291    4
Name: count, dtype: int64

### Examining one order

In [22]:
example_order = order_counts.index[0]

cultures.loc[
    cultures["order_proc_id_coded"] == example_order,
    [
        "anon_id",
        "culture_description",
        "was_positive",
        "organism",
        "antibiotic",
        "susceptibility"
    ]
].head(20)

,anon_id,culture_description,was_positive,organism,antibiotic,susceptibility
326,JC2881791,URINE,1,KLEBSIELLA PNEUMONIAE,Piperacillin/Tazobactam,Susceptible
3161,JC2881791,URINE,1,KLEBSIELLA PNEUMONIAE,Ciprofloxacin,Susceptible
5212,JC2881791,URINE,1,KLEBSIELLA PNEUMONIAE,Trimethoprim/Sulfamethoxazole,Susceptible
6010,JC2881791,URINE,1,KLEBSIELLA PNEUMONIAE,Tobramycin,Susceptible
6094,JC2881791,URINE,1,KLEBSIELLA PNEUMONIAE,Tetracycline,Susceptible


### Values for culture_description, organism, antibiotic, and susceptibility

In [25]:
columns = [
    "culture_description",
    "organism",
    "antibiotic",
    "susceptibility"
]

for column in columns:
    print(f"\n{column}:")
    print(cultures[column].unique())


culture_description:
<StringArray>
['URINE', 'BLOOD', 'RESPIRATORY']
Length: 3, dtype: str

organism:
<StringArray>
[                                          'KLEBSIELLA PNEUMONIAE',
                                           'RHIZOBIUM RADIOBACTER',
                                           'SERRATIA LIQUEFACIENS',
                                           'STAPHYLOCOCCUS AUREUS',
                                          'BREVIBACTERIUM SPECIES',
                                          'ENTEROBACTER GERGOVIAE',
                                          'MYCOBACTERIUM CHELONAE',
                                          'PROTEUS VULGARIS GROUP',
                                          'PSEUDOMONAS AERUGINOSA',
                                          'STAPHYLOCOCCUS SPECIES',
 ...
                  'PSEUDOMONAS SPECIES - FLUORESCENS/PUTIDA GROUP',
             'CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT)',
             'ENTEROBACTER CLOACAE COMPLEX (CARBAPENEM RESISTA

In [27]:
for column in columns:
    print(f"\n{column}:")
    
    for value in cultures[column].drop_duplicates().sort_values():
        print(value)


culture_description:
BLOOD
RESPIRATORY
URINE

organism:
ACHROMOBACTER SPECIES
ACHROMOBACTER XYLOSOXIDANS
ACINETOBACTER BAUMANNII
ACINETOBACTER BAUMANNII COMPLEX
ACINETOBACTER SPECIES
AEROMONAS HYDROPHILA GROUP
BACTEROIDES THETAIOTAOMICRON
BETA STREPTOCOCCUS GROUP G
BORDETELLA BRONCHISEPTICA
BREVIBACTERIUM SPECIES
BURKHOLDERIA CENOCEPACIA
BURKHOLDERIA CEPACIA COMPLEX GENOMOVAR I
BURKHOLDERIA GLADIOLI
BURKHOLDERIA MULTIVORANS
BURKHOLDERIA VIETNAMIENSIS
CAMPYLOBACTER SPECIES
CHRYSEOBACTERIUM INDOLOGENES
CHRYSEOBACTERIUM SPECIES
CITROBACTER AMALONATICUS COMPLEX
CITROBACTER FREUNDII COMPLEX
CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT)
CITROBACTER KOSERI (CARBAPENEM RESISTANT)
CLOSTRIDIUM CLOSTRIDIOFORME
CLOSTRIDIUM PERFRINGENS
COAG NEGATIVE STAPHYLOCOCCUS
CORYNEBACTERIUM ACCOLENS
CORYNEBACTERIUM DIPTHERIAE
CORYNEBACTERIUM JEIKEIUM
CORYNEBACTERIUM PROPINQUUM
CORYNEBACTERIUM PSEUDODIPHTHERITICUM
CORYNEBACTERIUM SPECIES
CORYNEBACTERIUM STRIATUM
CORYNEBACTERIUM UREALYTICUM
CRONOBACTER S

In [30]:
cultures["susceptibility"].value_counts(dropna=False)


susceptibility
Susceptible     7664
Resistant       1882
Intermediate     365
Inconclusive      41
Null              32
Synergism         16
Name: count, dtype: int64

In [32]:
null_results = cultures[
    cultures["susceptibility"] == "Null"
]

null_results[
    [
        "anon_id",
        "pat_enc_csn_id_coded",
        "order_proc_id_coded",
        "organism",
        "antibiotic",
        "susceptibility"
    ]
].head(10)

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,organism,antibiotic,susceptibility
1270,JC514481,131231684944,527334029,ENTEROBACTER CLOACAE COMPLEX,Ampicillin,Null
1271,JC568142,131236573032,531480637,ENTEROBACTER CLOACAE COMPLEX,Ampicillin,Null
1272,JC654450,131207275321,509843941,ENTEROBACTER CLOACAE COMPLEX,Ampicillin,Null
1498,JC2610004,131341720060,823747830,ACINETOBACTER BAUMANNII COMPLEX,Ceftazidime,Null
2914,JC2061792,131225998694,523353575,ZZZENTEROBACTER AEROGENES,Ampicillin,Null
3620,JC3252542,131283181938,651802315,STAPHYLOCOCCUS AUREUS,Nitrofurantoin,Null
3892,JC1816065,131328637352,781095490,PSEUDOMONAS PUTIDA GROUP,Ampicillin,Null
4117,JC558338,131357521244,887698689,COAG NEGATIVE STAPHYLOCOCCUS,Nitrofurantoin,Null
4118,JC2913941,131346333653,842646683,COAG NEGATIVE STAPHYLOCOCCUS,Nitrofurantoin,Null
4119,JC2032837,131300586465,698845144,COAG NEGATIVE STAPHYLOCOCCUS,Nitrofurantoin,Null


In [33]:
null_results["was_positive"].value_counts(dropna=False)

was_positive
1    32
Name: count, dtype: int64

### Does the same patient–order–organism–antibiotic combination appear more than once?

In [24]:
cultures.duplicated(
    subset=["anon_id", "order_proc_id_coded", "organism", "antibiotic"],
    keep=False
).sum()

np.int64(2)

In [34]:
duplicate_rows = cultures[
    cultures.duplicated(
        subset=["anon_id", "order_proc_id_coded", "organism", "antibiotic"],
        keep=False
    )
]

with pd.option_context("display.max_columns", None):
    display(duplicate_rows)

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,ordering_mode,culture_description,was_positive,organism,antibiotic,susceptibility
5433,JC1529500,131258547158,579935418,2018-10-30 05:10:00+00:00,Null,RESPIRATORY,1,STAPHYLOCOCCUS AUREUS,Erythromycin,Susceptible
8339,JC1529500,131258547158,579935418,2018-10-30 05:10:00+00:00,Null,RESPIRATORY,1,STAPHYLOCOCCUS AUREUS,Erythromycin,Resistant


### Conflicting susceptibility results

In the first 10,000 rows there is a discrepant susceptability. Two records share the same patient, order, organism, and antibiotic but have different susceptibility results:
Susceptible and Resistant.

The available columns do not establish whether these represent
different isolates, revised results, or another source-data issue.


### Duplicates?

In [35]:
cultures.duplicated().sum()

np.int64(0)

### Missing values

In [36]:
cultures.isna().sum()

anon_id                    0
pat_enc_csn_id_coded       0
order_proc_id_coded        0
order_time_jittered_utc    0
ordering_mode              0
culture_description        0
was_positive               0
organism                   0
antibiotic                 0
susceptibility             0
dtype: int64

In [37]:
cultures.eq("Null").sum()

anon_id                      0
pat_enc_csn_id_coded         0
order_proc_id_coded          0
order_time_jittered_utc      0
ordering_mode              985
culture_description          0
was_positive                 0
organism                     0
antibiotic                   0
susceptibility              32
dtype: int64

In [38]:
cultures["was_positive"].value_counts(dropna=False)

was_positive
1    10000
Name: count, dtype: int64

### Missing values and placeholders

No missing values in the 10,000 sample were detected.
THere are 985 literal string "Null" values in ordering_mode and 32 susceptibility values

## Demographics dataset

Inspect the first 10,000 rows of the demographics dataset 'microbiology_cultures_demographics.csv'

In [39]:
demographics = pd.read_csv(
    data_dir / "microbiology_cultures_demographics.csv",
    nrows=10_000
)

demographics.head()

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,age,gender
0,JC1212710,131008781669,364609652,25-34 years,Null
1,JC1218441,33052060,351667304,25-34 years,Null
2,JC1261412,131004002288,355531114,25-34 years,Null
3,JC1224853,19532426,335118697,25-34 years,Null
4,JC1242725,32587123,349578403,25-34 years,Null


In [40]:

demographics.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   anon_id               10000 non-null  str  
 1   pat_enc_csn_id_coded  10000 non-null  int64
 2   order_proc_id_coded   10000 non-null  int64
 3   age                   10000 non-null  str  
 4   gender                10000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 390.8 KB


In [41]:
demographics.columns.tolist()

['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'age', 'gender']

### Count distinct identifiers

In [42]:
demographics[
    ["anon_id", "pat_enc_csn_id_coded", "order_proc_id_coded"]
].nunique()

anon_id                  4131
pat_enc_csn_id_coded     7400
order_proc_id_coded     10000
dtype: int64

### Check for repeated combinations of all three identifiers

In [44]:
demographics.duplicated(
    subset=["anon_id", "pat_enc_csn_id_coded", "order_proc_id_coded"],
    keep=False
).sum()

np.int64(0)

In [45]:
demographics["age"].value_counts(dropna=False)

age
18-24 years    9994
25-34 years       6
Name: count, dtype: int64

In [46]:
demographics["gender"].value_counts(dropna=False)

gender
1       9994
Null       6
Name: count, dtype: int64

In [47]:
demographics_full = pd.read_csv(
    data_dir / "microbiology_cultures_demographics.csv"
)

demographics_full.shape

C:\Users\clark\AppData\Local\Temp\ipykernel_149552\1763473746.py:1: DtypeWarning: Columns (4: gender) have mixed types. Specify dtype option on import or set low_memory=False.
  demographics_full = pd.read_csv(


(751075, 5)

In [48]:
demographics_full["age"].value_counts(dropna=False)

age
65-74 years    130764
55-64 years    122486
75-84 years    100915
25-34 years     99023
35-44 years     93245
45-54 years     93033
18-24 years     47509
85-89 years     34458
above 90        29642
Name: count, dtype: int64

In [49]:
demographics_full["gender"].value_counts(dropna=False)

gender
0       383213
1       141075
1       131066
0        95601
Null       120
Name: count, dtype: int64

In [50]:
for value in demographics_full["gender"].unique():
    print(repr(value), type(value).__name__)

'Null' str
'1' str
1 int
0 int
'0' str


In [51]:
demographics_full = pd.read_csv(
    data_dir / "microbiology_cultures_demographics.csv",
    dtype={"gender": "string"}
)

In [52]:
demographics_full["gender"].value_counts(dropna=False)

gender
0       478814
1       272141
Null       120
Name: count, dtype: Int64

### Pandas inferred mixed types for gender. I explicitly load this column as text to keep the codes consistent.

### 

### There are strings and numbers present for gender values. Must use dtype when importing

## Ward dataset

Inspect the full ward dataset: 'microbiology_cultures_ward_info.csv'

In [53]:
wards = pd.read_csv(
    data_dir / "microbiology_cultures_ward_info.csv"
)

wards.shape

(751075, 8)

In [54]:
wards.head()

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,hosp_ward_IP,hosp_ward_OP,hosp_ward_ER,hosp_ward_ICU
0,JC1992981,131188197629,496076158,2016-07-02 21:08:00+00:00,0,1,0,0
1,JC542594,15917502,327566424,2008-07-16 03:23:00+00:00,0,1,0,0
2,JC597158,131276794493,642698667,2019-11-10 22:54:00+00:00,0,1,0,0
3,JC1579203,131319187199,751697107,2021-10-29 07:00:00+00:00,0,1,0,0
4,JC534140,131027275219,427106380,2013-09-05 04:53:00+00:00,0,1,0,0


In [55]:
wards.info()

<class 'pandas.DataFrame'>
RangeIndex: 751075 entries, 0 to 751074
Data columns (total 8 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   anon_id                  751075 non-null  str  
 1   pat_enc_csn_id_coded     751075 non-null  int64
 2   order_proc_id_coded      751075 non-null  int64
 3   order_time_jittered_utc  751075 non-null  str  
 4   hosp_ward_IP             751075 non-null  int64
 5   hosp_ward_OP             751075 non-null  int64
 6   hosp_ward_ER             751075 non-null  int64
 7   hosp_ward_ICU            751075 non-null  int64
dtypes: int64(6), str(2)
memory usage: 45.8 MB


In [56]:
ward_columns = [
    "hosp_ward_IP",
    "hosp_ward_OP",
    "hosp_ward_ER",
    "hosp_ward_ICU"
]

for column in ward_columns:
    print(f"\n{column}:")
    print(wards[column].value_counts(dropna=False))


hosp_ward_IP:
hosp_ward_IP
1    464113
0    286962
Name: count, dtype: int64

hosp_ward_OP:
hosp_ward_OP
0    475771
1    275304
Name: count, dtype: int64

hosp_ward_ER:
hosp_ward_ER
0    460846
1    290229
Name: count, dtype: int64

hosp_ward_ICU:
hosp_ward_ICU
0    700462
1     50613
Name: count, dtype: int64


In [57]:
wards[ward_columns].sum(axis=1).value_counts().sort_index()

0     11658
1    427739
2    282514
3     29164
Name: count, dtype: int64

In [58]:
wards[ward_columns].value_counts(
    dropna=False
).reset_index(name="row_count")

,hosp_ward_IP,hosp_ward_OP,hosp_ward_ER,hosp_ward_ICU,row_count
0,0,1,0,0,275296
1,1,0,1,0,261060
2,1,0,0,0,152443
3,1,0,1,1,29164
4,1,0,0,1,21446
5,0,0,0,0,11658
6,0,1,1,0,5
7,0,1,0,1,3


## Check Joins demographics and ward tables

In [60]:
join_keys = [
    "anon_id",
    "pat_enc_csn_id_coded",
    "order_proc_id_coded"
]

In [61]:
demographics_full.duplicated(
    subset=join_keys,
    keep=False
).sum()

np.int64(0)

In [62]:
wards.duplicated(
    subset=join_keys,
    keep=False
).sum()

np.int64(0)

In [63]:
key_check = demographics_full[join_keys].merge(
    wards[join_keys],
    on=join_keys,
    how="outer",
    indicator=True,
    validate="one_to_one"
)

key_check["_merge"].value_counts()

_merge
both          751075
left_only          0
right_only         0
Name: count, dtype: int64

In [64]:
order_details = demographics_full.merge(
    wards,
    on=join_keys,
    how="left",
    validate="one_to_one"
)

order_details.shape

(751075, 10)

In [65]:
with pd.option_context("display.max_columns", None):
    display(order_details.head())

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,age,gender,order_time_jittered_utc,hosp_ward_IP,hosp_ward_OP,hosp_ward_ER,hosp_ward_ICU
0,JC1212710,131008781669,364609652,25-34 years,Null,2010-03-05 00:25:00+00:00,0,1,0,0
1,JC1218441,33052060,351667304,25-34 years,Null,2009-05-13 00:09:00+00:00,0,1,0,0
2,JC1261412,131004002288,355531114,25-34 years,Null,2009-09-06 22:29:00+00:00,0,1,0,0
3,JC1224853,19532426,335118697,25-34 years,Null,2008-12-07 23:22:00+00:00,0,1,0,0
4,JC1242725,32587123,349578403,25-34 years,Null,2009-05-04 03:53:00+00:00,0,1,0,0


## Check cultures table joins

In [66]:
detail_columns = join_keys + ["age", "gender"] + ward_columns

cultures_enriched = cultures.merge(
    order_details[detail_columns],
    on=join_keys,
    how="left",
    validate="many_to_one",
    indicator=True
)

In [69]:
cultures_enriched["_merge"].value_counts()
print("Before join:", len(cultures))
print("After join:", len(cultures_enriched))

Before join: 10000
After join: 10000


In [70]:
cultures_enriched["_merge"].value_counts()

_merge
both          10000
left_only         0
right_only        0
Name: count, dtype: int64

In [71]:
cultures_full = pd.read_csv(
    data_dir / "microbiology_cultures_cohort.csv",
    dtype={
        "ordering_mode": "string",
        "organism": "string",
        "antibiotic": "string",
        "susceptibility": "string"
    }
)

cultures_full.shape

(2241050, 10)

In [72]:
cultures_full["was_positive"].value_counts(dropna=False)

was_positive
1    1610252
0     630798
Name: count, dtype: int64

In [73]:
cultures_full.loc[
    cultures_full["was_positive"] == 0,
    [
        "culture_description",
        "was_positive",
        "organism",
        "antibiotic",
        "susceptibility"
    ]
].head(10)

,culture_description,was_positive,organism,antibiotic,susceptibility
29824,BLOOD,0,Null,Null,Null
29825,URINE,0,Null,Null,Null
29826,URINE,0,Null,Null,Null
29827,URINE,0,Null,Null,Null
29828,BLOOD,0,Null,Null,Null
29829,RESPIRATORY,0,Null,Null,Null
29830,URINE,0,Null,Null,Null
29831,BLOOD,0,Null,Null,Null
29832,URINE,0,Null,Null,Null
29833,RESPIRATORY,0,Null,Null,Null


In [74]:
negative_rows = cultures_full[
    cultures_full["was_positive"] == 0
]

negative_rows[
    ["organism", "antibiotic", "susceptibility"]
].value_counts(dropna=False)

organism  antibiotic  susceptibility
Null      Null        Null              630798
Name: count, dtype: int64

In [75]:
negative_rows.duplicated(
    subset=join_keys,
    keep=False
).sum()

np.int64(0)

In [76]:
culture_orders = cultures_full[join_keys].drop_duplicates()

culture_orders.shape

(751075, 3)

In [77]:
order_match_check = culture_orders.merge(
    order_details[join_keys],
    on=join_keys,
    how="outer",
    indicator=True,
    validate="one_to_one"
)

order_match_check["_merge"].value_counts()

_merge
both          751075
left_only          0
right_only         0
Name: count, dtype: int64

In [78]:
cultures_full.duplicated().sum()

np.int64(0)

In [79]:
result_keys = join_keys + ["organism", "antibiotic"]

susceptibility_counts = (
    cultures_full
    .groupby(result_keys, dropna=False)["susceptibility"]
    .nunique(dropna=False)
)

(susceptibility_counts > 1).sum()

np.int64(9047)

In [80]:
conflict_keys = susceptibility_counts[
    susceptibility_counts > 1
].reset_index(name="distinct_results")

conflicting_rows = cultures_full.merge(
    conflict_keys,
    on=result_keys,
    how="inner",
    validate="many_to_one"
)

In [81]:
len(conflicting_rows)

18424

In [82]:
with pd.option_context("display.max_columns", None):
    display(
        conflicting_rows
        .sort_values(result_keys)
        .head(10)
    )

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,ordering_mode,culture_description,was_positive,organism,antibiotic,susceptibility,distinct_results
1676,JC1000169,131216209091,516543570,2017-01-12 04:40:00+00:00,Outpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Nitrofurantoin,Intermediate,2
12429,JC1000169,131216209091,516543570,2017-01-12 04:40:00+00:00,Outpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Nitrofurantoin,Susceptible,2
842,JC1002322,131292736345,676541157,2020-08-01 20:44:00+00:00,Inpatient,URINE,1,PSEUDOMONAS AERUGINOSA,Ciprofloxacin,Susceptible,2
12127,JC1002322,131292736345,676541157,2020-08-01 20:44:00+00:00,Inpatient,URINE,1,PSEUDOMONAS AERUGINOSA,Ciprofloxacin,Resistant,2
10930,JC1002322,131292736345,676541157,2020-08-01 20:44:00+00:00,Inpatient,URINE,1,PSEUDOMONAS AERUGINOSA,Levofloxacin,Susceptible,2
16079,JC1002322,131292736345,676541157,2020-08-01 20:44:00+00:00,Inpatient,URINE,1,PSEUDOMONAS AERUGINOSA,Levofloxacin,Resistant,2
2923,JC1005958,131011786120,376367046,2010-12-19 17:31:00+00:00,Outpatient,URINE,1,ESCHERICHIA COLI,Ampicillin/Sulbactam,Intermediate,2
7051,JC1005958,131011786120,376367046,2010-12-19 17:31:00+00:00,Outpatient,URINE,1,ESCHERICHIA COLI,Ampicillin/Sulbactam,Resistant,2
6352,JC1006285,131332557851,794117141,2022-06-09 20:02:00+00:00,Outpatient,URINE,1,ESCHERICHIA COLI,Cefoxitin,Susceptible,2
15472,JC1006285,131332557851,794117141,2022-06-09 20:02:00+00:00,Outpatient,URINE,1,ESCHERICHIA COLI,Cefoxitin,Intermediate,2


In [83]:
conflict_patterns = (
    conflicting_rows
    .groupby(result_keys, dropna=False)["susceptibility"]
    .agg(
        lambda results: " + ".join(
            sorted(results.fillna("<missing>").unique())
        )
    )
)

conflict_patterns.value_counts()

susceptibility
Resistant + Susceptible                                  3985
Intermediate + Susceptible                               2097
Intermediate + Resistant                                 1338
Inconclusive + Resistant                                  780
Inconclusive + Susceptible                                290
Intermediate + Resistant + Susceptible                    201
Null + Susceptible                                        127
Inconclusive + Resistant + Susceptible                     86
Inconclusive + Intermediate                                68
Null + Resistant                                           28
Inconclusive + Intermediate + Resistant                    28
Intermediate + Null                                         9
Inconclusive + Intermediate + Resistant + Susceptible       5
Inconclusive + Null + Susceptible                           1
Inconclusive + Null                                         1
Inconclusive + Intermediate + Null                     

In [84]:
print(
    "Orders with multiple susceptibility values:",
    len(conflicting_rows[join_keys].drop_duplicates())
)

print(
    "Patients with multiple susceptibility values:",
    conflicting_rows["anon_id"].nunique()
)

Orders with multiple susceptibility values: 4400
Patients with multiple susceptibility values: 2929


In [85]:
order_fields = [
    "order_time_jittered_utc",
    "ordering_mode",
    "culture_description",
    "was_positive"
]

order_field_counts = (
    cultures_full
    .groupby(join_keys, dropna=False)[order_fields]
    .nunique(dropna=False)
)

(order_field_counts > 1).sum()

order_time_jittered_utc    0
ordering_mode              0
culture_description        0
was_positive               0
dtype: int64

In [86]:
order_summary = (
    cultures_full[join_keys + order_fields]
    .drop_duplicates()
    .copy()
)

order_summary.shape

(751075, 7)

In [87]:
order_summary["order_time_parsed"] = pd.to_datetime(
    order_summary["order_time_jittered_utc"],
    utc=True,
    errors="coerce"
)

order_summary["order_time_parsed"].isna().sum()

np.int64(0)

In [88]:
wards["ward_time_parsed"] = pd.to_datetime(
    wards["order_time_jittered_utc"],
    utc=True,
    errors="coerce"
)

wards["ward_time_parsed"].isna().sum()

np.int64(0)

In [89]:
timestamp_check = order_summary.merge(
    wards[join_keys + ["ward_time_parsed"]],
    on=join_keys,
    how="left",
    validate="one_to_one"
)

(
    timestamp_check["order_time_parsed"]
    != timestamp_check["ward_time_parsed"]
).sum()

np.int64(0)

### Timestamps parse successfully and agree between files.

In [90]:
pd.crosstab(
    order_summary["culture_description"],
    order_summary["was_positive"],
    margins=True
)

was_positive,0,1,All
culture_description,,,
BLOOD,278315,13050,291365
RESPIRATORY,66699,17834,84533
URINE,285784,89393,375177
All,630798,120277,751075


## Initial findings

These findings use the full files from the April 11, 2025 dataset version.

### Dataset structure

The culture file contains 2,241,050 rows representing 751,075 distinct
patient–encounter–order combinations.

The demographics and ward files each contain one row per combination.
All identifiers match across the three files, with no unmatched orders.

| Culture category | was_positive = 0 | was_positive = 1 | Total orders |
|---|---:|---:|---:|
| Blood | 278,315 | 13,050 | 291,365 |
| Respiratory | 66,699 | 17,834 | 84,533 |
| Urine | 285,784 | 89,393 | 375,177 |
| Total | 630,798 | 120,277 | 751,075 |

These counts describe the released dataset using its source positivity flag.

### Data quality observations

- No exact duplicate rows were found in the culture file.
- All 630,798 rows marked was_positive = 0 contain the text "Null"
  for organism, antibiotic, and susceptibility.
- Multiple susceptibility values occur in 9,047 patient–encounter–order–
  organism–antibiotic groups. These involve 18,424 rows, 4,400 orders,
  and 2,929 patients. The cause remains unresolved; records are retained.
- Order timestamps, ordering mode, culture description, and positivity
  flags are consistent within each order.
- Order timestamps parse successfully and agree between cultures and wards.
- Ward indicators overlap: 311,678 rows have multiple flags set,
  while 11,658 have none.
- Gender requires an explicit text data type to prevent mixed-type
  inference. The meanings of codes 0 and 1 still need confirmation.
- Age is supplied in groups rather than exact years.

### Implications for the pipeline

- Preserve the original source values.
- Separate order-level information from organism–antibiotic results.
- Validate joins to prevent accidental multiplication of rows.
- Flag ambiguous susceptibility combinations for explicit handling.
- Preserve the separate ward indicators.